In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd


e:\AI\Project\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4666.68it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
movies = pd.read_csv('../data/movies_5000.csv') 

In [4]:
movies['content'] = movies['overview']+ movies['tag']

In [5]:
# Weighted content without description
movies['content'] = (
    movies['title']*3 +      # give highest weight to title
    movies['overview']*2 +   # medium weight to overview
    movies['tag']            # normal weight to tags
)

In [6]:
movie_embeddings = model.encode(movies['content'].tolist(), show_progress_bar=True)


Batches: 100%|██████████| 151/151 [03:02<00:00,  1.21s/it]


In [7]:
np.save('movie_embeddings.npy', movie_embeddings)

In [8]:
movie_embeddings = np.load('movie_embeddings.npy')

In [9]:
def clean_input(text):
    noise_words = ['movie', 'movies', 'film', 'films', 'cinema']
    
    text = text.lower()
    words = text.split()
    
    cleaned_words = [word for word in words if word not in noise_words]
    
    return " ".join(cleaned_words)


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_hybrid(user_input, top_n=5):
    # 🔥 Clean input first
    user_input = clean_input(user_input)

    # 1️⃣ Exact title match first
    exact_matches = movies[movies['title'].str.contains(user_input, case=False, na=False)]
    if len(exact_matches) >= top_n:
        return exact_matches.head(top_n)[['title']]
    
    # 2️⃣ Semantic search
    user_embedding = model.encode([user_input])
    similarity = cosine_similarity(user_embedding, movie_embeddings)[0]
    top_indices = np.argsort(similarity)[::-1][:top_n]
    
    return movies.iloc[top_indices][['title']]

In [11]:
print(recommend_hybrid ("movies about motivation inspiring movies", top_n=5))

                                  title
1680                    United Passions
493                    A Beautiful Mind
1513  Dreamer: Inspired By a True Story
3247                              50/50
1326                        The Express
